In [ ]:
###!/usr/bin/env python
################################################
# New style 
# ###############################################
import sys

rootdir_ = '../'
if ( rootdir_ not in sys.path ):
    sys.path.append(rootdir_)
    print( f" a path to {rootdir_} added in {__name__} ")


from Utils import GridUtils as GrU
from Utils import MakePressures as MkP
from Utils import utils as uti
from Utils import MyConstants as Co
from Utils import time_utils as tuti
from Utils import numerical_utils as nuti

import analysis_utils as auti
import file_utils as futi
import event_utils as euti


#from PyRegridding.Utils import MakePressures as MkP
#from Drivers import RegridField as RgF
import RegridField as RgF

# The usual
from datetime import date
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# for smoothing , nonlienar colors ...
from scipy.ndimage import uniform_filter
from scipy.ndimage import gaussian_filter
import matplotlib.colors as mcolors

# Some other useful packages 
import copy
import time
import cftime
import yaml
import numbers

# Some other useful packages 
import importlib
from pathlib import Path


importlib.reload( auti )
importlib.reload( futi )
importlib.reload( euti )

Rdair=Co.Rdair()


In [ ]:
# This allow both dict.key and dict['key'] syntax
class AttrDict(dict):
    def __getattr__(self, key):
        try:
            return self[key]
        except KeyError:
            raise AttributeError(f"'AttrDict' object has no attribute '{key}'")

    def __setattr__(self, key, value):
        self[key] = value

    def __delattr__(self, key):
        try:
            del self[key]
        except KeyError:
            raise AttributeError(f"'AttrDict' object has no attribute '{key}'")



In [ ]:
%%time
nsteps=None
start_date=None
super_lat_range = [-90.,90.]  #[-85,-30]
#case, process_ncdata, start_date, nsteps = 'c153_topfix_ne240pg3_FMTHIST_xic_x02'   , False, [2004,7,15,0], 248
case, process_ncdata, nsteps  = 'cam77_dyamond1_prod1'    , False, 128
#case , process_ncdata = 'xy-rdg-mm-front'    , True
A = futi.read_case( case=case, nsteps=nsteps, start_date=start_date , super_lat_range=super_lat_range ) # , nsteps = 31*8 )

time, zlev, lat, lon = A.time, A.zlev, A.lat, A.lon



In [ ]:
#################################################################
# Make event lists ... and composites
importlib.reload(euti)
importlib.reload(auti)

thresh=0.01 #0.005 # 0.02
thresh=0.001 #0.005 # 0.02
thresh=0.0001 #0.005 # 0.02
zlev_event=10_000. #23_000.
zlev_event=15_000. #23_000.

El=[]

print( f"This run use dycore={A.dycore}")

if A.dycore == 'MPAS':
    ## for MPAS 3km
    thresholds=[  [0.005,1e6], 
                [0.008,1e6],
                [0.01,1e6] ,
                [0.020,1e6]  ] ##[0.025,1e6]  ] ##0.025, ]
    second_thresholds= [
                    [0.005,0.008], 
                    [0.008,0.015],
                    [0.01,0.025] ,
                    [0.020,1e6]  ] ## [0.025,1e6]  ] ##0.025, ]
elif A.dycore == 'SE':
    # for ne240
    thresholds=[  [0.0005,1e6], 
                [0.002,1e6],
                [0.005,1e6] ,
                [0.010,1e6]  ] ##[0.025,1e6]  ] ##0.025, ]
    second_thresholds= [
                    [0.0005,0.002], 
                    [0.002,0.0075],
                    [0.005,0.012] ,
                    [0.010,1e6]  ] ## [0.025,1e6]  ] ##0.025, ]

lat_range=  [-65,-40] #[-70,-60] #[-60,-40]
lon_range=[0,60] # [0,60]
lat_range=  [-50,-40] #[-70,-60] #[-60,-40]
lat_range=  [-60,-50] #[-70,-60] #[-60,-40]
lon_range=[0,360] # [0,60]

ithr=0
for thresh in thresholds:
    second_thresh=second_thresholds[ithr]
    ds =euti.make_ds(fld=A.rho_epwp[:,:,:,:], lon=lon, lat=lat, zlev=zlev, time=time, 
                     thresh=thresh,second_thresh=second_thresh,zlev_event=zlev_event, 
                     lat_range=lat_range, lon_range=lon_range)
    ithr=ithr+1
    

    # get shape of varaiables
    nt,nz,ny,nx = np.shape( A.u )
    
    htopo_t = np.tile(A.htopo[None, :, :], ( nt, 1, 1))
    htopo_4D_x , time4D,lat4D,lon4D  = auti.cube4D_ds( event_ds=ds, aa=htopo_t , lon=lon, lat=lat, window=[0,5,5] , TZHkey='tyx', lat_range=[-999,999], lon_range=[-999,999] )
    
    htopo_MMM = auti.collapseSpace( htopo_4D_x , TZHkey='etyx')
    htopo_super_max = htopo_MMM[2].max( axis=1 )
    
    
    ########################################
    # Exclude events with topography nearby
    ########################################
    
    flat=np.where(htopo_super_max<0.0001)
    flat[0].shape
    
    ds_flat=ds.isel( index=flat[0] )    
    ds=ds_flat
    
    E_ = {'ds':ds }

    window=[3,2,2]
    #window=[6,2,2] # MPAS results are 3-hourly ...
    if ds.sizes['index'] < 75_000:
        window = [3,5,5] #[6,5,5]
        print( f"Big window ")
    #window=[6,5,5] # MPAS results are 3-hourly ...
    
    
    
    precl_4D , time4D,lat4D,lon4D  = auti.cube4D_ds( event_ds=ds, aa=A.precl , lon=lon, lat=lat, window=window , TZHkey='tzyx', lat_range=lat_range, lon_range=lon_range )
    htopo_4D , time4D,lat4D,lon4D  = auti.cube4D_ds( event_ds=ds, aa=htopo_t , lon=lon, lat=lat, window=window , TZHkey='tyx', lat_range=lat_range, lon_range=lon_range )
    zeta_4D, time4D,lat4D,lon4D = auti.cube4D_ds( event_ds=ds, aa=A.zeta , lon=lon, lat=lat, window=window , TZHkey='tzyx', lat_range=lat_range, lon_range=lon_range )
    tilt_4D, time4D,lat4D,lon4D = auti.cube4D_ds( event_ds=ds, aa=A.tilt , lon=lon, lat=lat, window=window , TZHkey='tzyx', lat_range=lat_range, lon_range=lon_range )
    fgf_4D, time4D,lat4D,lon4D  = auti.cube4D_ds( event_ds=ds, aa=A.fgf , lon=lon, lat=lat, window=window , TZHkey='tzyx', lat_range=lat_range, lon_range=lon_range )
    epwp_4D, time4D,lat4D,lon4D = auti.cube4D_ds( event_ds=ds, aa=A.rho_epwp , lon=lon, lat=lat, window=window , TZHkey='tzyx', lat_range=lat_range, lon_range=lon_range )
    upwp_4D, time4D,lat4D,lon4D = auti.cube4D_ds( event_ds=ds, aa=A.rho_upwp , lon=lon, lat=lat, window=window , TZHkey='tzyx', lat_range=lat_range, lon_range=lon_range )
    vpwp_4D, time4D,lat4D,lon4D = auti.cube4D_ds( event_ds=ds, aa=A.rho_vpwp , lon=lon, lat=lat, window=window , TZHkey='tzyx', lat_range=lat_range, lon_range=lon_range )
    u_4D, time4D,lat4D,lon4D    = auti.cube4D_ds( event_ds=ds, aa=A.u , lon=lon, lat=lat, window=window , TZHkey='tzyx', lat_range=lat_range, lon_range=lon_range )
    v_4D, time4D,lat4D,lon4D    = auti.cube4D_ds( event_ds=ds, aa=A.v , lon=lon, lat=lat, window=window , TZHkey='tzyx', lat_range=lat_range, lon_range=lon_range )
    th_4D, time4D,lat4D,lon4D   = auti.cube4D_ds( event_ds=ds, aa=A.th , lon=lon, lat=lat, window=window , TZHkey='tzyx', lat_range=lat_range, lon_range=lon_range )

    E_['time4D'], E_['lat4D'], E_['lon4D'] = time4D,lat4D,lon4D
    E_['u_4D'], E_['v_4D'], E_['htopo_4D'] = u_4D,v_4D,htopo_4D
    E_['zeta_4D'], E_['tilt_4D'] , E_['fgf_4D']  = zeta_4D,tilt_4D,fgf_4D
    E_['upwp_4D'], E_['vpwp_4D'] , E_['epwp_4D']  = upwp_4D,vpwp_4D,epwp_4D
    E_['precl_4D'], E_['th_4D'] = precl_4D,th_4D

    E = AttrDict( E_ )
    El.append(E)
    print( f" fininshed threshold = {thresh}. N events: Global= {E.ds.sizes['index']}, Region= {len(time4D)} " )

In [ ]:
zetalv=1.5e-5*np.linspace(-6,6,num=13)
ulv=np.linspace(-60,60,num=27)
thlv=np.concatenate( (270.+np.arange(11)*10 , 380.+ np.arange(11)*20) )   #np.linspace(270,600,num=27)
mflv=[0.001,0.002,.005, .01, .02]
Epls=[El[0], El[1], El[2], El[3] ] #, Eco_0 ]
print(thlv)
nxplo,nyplo=len( Epls ),1
fig,axs=plt.subplots( nyplo,nxplo , figsize=(nxplo*7+1,nyplo*8) )
axs=axs.flatten()
p=0

for Epl in Epls:
    nv,nt_v,nz_v,ny_v,nx_v = np.shape( Epl.zeta_4D )
    delta_time=3

    ax=axs[p]
    Epl_vv=euti.avg_over_v(Epl)
    zeta_poo=Epl_vv.zeta_4D.mean(axis=3)
    u_poo=Epl_vv.u_4D.mean(axis=3)
    th_poo=Epl_vv.th_4D.mean(axis=3)
    epwp_poo=Epl_vv.epwp_4D.mean(axis=3)
    colo = ax.contourf( np.arange(ny_v), zlev, zeta_poo[3,:,:], cmap='bwr' , levels=zetalv)
    lin1 = ax.contour( np.arange(ny_v), zlev, u_poo[3,:,:] , levels=ulv)
    ax.clabel(lin1, inline=True, fontsize=8, fmt='%1.0f')
    lin2 = ax.contour( np.arange(ny_v), zlev, th_poo[3,:,:] , levels=thlv, colors='red')
    ax.clabel(lin2, inline=True, fontsize=8, fmt='%1.0f')
    lin3 = ax.contour( np.arange(ny_v), zlev, epwp_poo[3,:,:] , levels=mflv, colors='black')
    ax.clabel(lin3, inline=True, fontsize=8, fmt='%1.3f')
    ax.set_ylim(0,20_000)
    p=p+1

cax = fig.add_axes([0.15, 0.02, 0.70, 0.03])
cbar = fig.colorbar(colo, cax=cax, orientation='horizontal')
cbar.set_label(f"vorticity  s{r'$^{-1}$' }")
plt.suptitle(  " Global Latitude ",fontsize=36 )
"""
cax = fig.add_axes([0.15, -0.1, 0.70, 0.03])
cbar = fig.colorbar(linc, cax=cax, orientation='horizontal')
cbar.set_label('Some quantity')
"""

In [ ]:
zetalv=0.75e-7*np.linspace(0,6,num=31)
ulv=np.linspace(-60,60,num=27)
thlv=np.concatenate( (270.+np.arange(11)*10 , 380.+ np.arange(11)*20) )   #np.linspace(270,600,num=27)
Epls=[El[0], El[1], El[2], El[3] ] #, Eco_0 ]
print(thlv)
nxplo,nyplo=len( Epls ),1
fig,axs=plt.subplots( nyplo,nxplo , figsize=(nxplo*7+1,nyplo*8) )
axs=axs.flatten()
p=0

for Epl in Epls:
    nv,nt_v,nz_v,ny_v,nx_v = np.shape( Epl.zeta_4D )
    delta_time=3

    ax=axs[p]
    Epl_vv=euti.avg_over_v(Epl)
    tilt_poo=Epl_vv.tilt_4D.mean(axis=3)
    u_poo=Epl_vv.u_4D.mean(axis=3)
    th_poo=Epl_vv.th_4D.mean(axis=3)
    colo = ax.contourf( np.arange(ny_v), zlev, tilt_poo[3,:,:], cmap='hot' , levels=zetalv)
    lin1 = ax.contour( np.arange(ny_v), zlev, u_poo[3,:,:] , levels=ulv)
    ax.clabel(lin1, inline=True, fontsize=8, fmt='%1.0f')
    lin2 = ax.contour( np.arange(ny_v), zlev, th_poo[3,:,:] , levels=thlv, colors='red')
    ax.clabel(lin2, inline=True, fontsize=8, fmt='%1.0f')
    ax.set_ylim(0,20_000)
    p=p+1

cax = fig.add_axes([0.15, 0.02, 0.70, 0.03])
cbar = fig.colorbar(colo, cax=cax, orientation='horizontal')
cbar.set_label(f"Tilting  s{r'$^{-2}$' }")
"""
cax = fig.add_axes([0.15, -0.1, 0.70, 0.03])
cbar = fig.colorbar(linc, cax=cax, orientation='horizontal')
cbar.set_label('Some quantity')
"""

In [ ]:
zetalv=1.5e-15*np.linspace(-6,6,num=31)
ulv=np.linspace(-60,60,num=27)
thlv=np.concatenate( (270.+np.arange(11)*10 , 380.+ np.arange(11)*20) )   #np.linspace(270,600,num=27)
Epls=[El[0], El[1], El[2], El[3] ] #, Eco_0 ]
print(thlv)
nxplo,nyplo=len( Epls ),1
fig,axs=plt.subplots( nyplo,nxplo , figsize=(nxplo*7+1,nyplo*8) )
axs=axs.flatten()
p=0

for Epl in Epls:
    nv,nt_v,nz_v,ny_v,nx_v = np.shape( Epl.zeta_4D )
    delta_time=3

    ax=axs[p]
    Epl_vv=euti.avg_over_v(Epl)
    tilt_poo=Epl_vv.fgf_4D.mean(axis=3)
    u_poo=Epl_vv.u_4D.mean(axis=3)
    th_poo=Epl_vv.th_4D.mean(axis=3)
    colo = ax.contourf( np.arange(ny_v), zlev, tilt_poo[3,:,:], cmap='bwr' , levels=zetalv)
    lin1 = ax.contour( np.arange(ny_v), zlev, u_poo[3,:,:] , levels=ulv)
    ax.clabel(lin1, inline=True, fontsize=8, fmt='%1.0f')
    lin2 = ax.contour( np.arange(ny_v), zlev, th_poo[3,:,:] , levels=thlv, colors='red')
    ax.clabel(lin2, inline=True, fontsize=8, fmt='%1.0f')
    ax.set_ylim(0,20_000)
    p=p+1

cax = fig.add_axes([0.15, 0.02, 0.70, 0.03])
cbar = fig.colorbar(colo, cax=cax, orientation='horizontal')
cbar.set_label(f"Frontogenesis  s{r'$^{-2}$' }")
"""
cax = fig.add_axes([0.15, -0.1, 0.70, 0.03])
cbar = fig.colorbar(linc, cax=cax, orientation='horizontal')
cbar.set_label('Some quantity')
"""

In [ ]:
print(poopypants)

In [ ]:
A_0=copy.deepcopy(A)
El_0=copy.deepcopy(El)

In [ ]:
del A,El

In [ ]:
%%time
nsteps=None
start_date=None
super_lat_range = [-80,-30]
#case, process_ncdata, start_date, nsteps = 'c153_topfix_ne240pg3_FMTHIST_xic_x02'   , False, [2004,7,15,0], 248
case, process_ncdata, nsteps  = 'cam77_dyamond1_prod1'    , False, 128
#case , process_ncdata = 'xy-rdg-mm-front'    , True
A = futi.read_case( case=case, nsteps=nsteps, start_date=start_date , super_lat_range=super_lat_range ) # , nsteps = 31*8 )

time, zlev, lat, lon = A.time, A.zlev, A.lat, A.lon



In [ ]:
print(A.u.shape)

In [ ]:
#################################################################
# Make event lists ... and composites
importlib.reload(euti)
importlib.reload(auti)

thresh=0.01 #0.005 # 0.02
thresh=0.001 #0.005 # 0.02
thresh=0.0001 #0.005 # 0.02
zlev_event=10_000. #23_000.
zlev_event=15_000. #23_000.

El=[]

print( f"This run use dycore={A.dycore}")

if A.dycore == 'MPAS':
    ## for MPAS 3km
    thresholds=[  [0.005,1e6], 
                [0.008,1e6],
                [0.01,1e6] ,
                [0.020,1e6]  ] ##[0.025,1e6]  ] ##0.025, ]
    second_thresholds= [
                    [0.005,0.008], 
                    [0.008,0.015],
                    [0.01,0.025] ,
                    [0.020,1e6]  ] ## [0.025,1e6]  ] ##0.025, ]
elif A.dycore == 'SE':
    # for ne240
    thresholds=[  [0.0005,1e6], 
                [0.002,1e6],
                [0.005,1e6] ,
                [0.010,1e6]  ] ##[0.025,1e6]  ] ##0.025, ]
    second_thresholds= [
                    [0.0005,0.002], 
                    [0.002,0.0075],
                    [0.005,0.012] ,
                    [0.010,1e6]  ] ## [0.025,1e6]  ] ##0.025, ]

lat_range=  [-65,-40] #[-70,-60] #[-60,-40]
lon_range=[0,60] # [0,60]
lat_range=  [-50,-40] #[-70,-60] #[-60,-40]
lat_range=  [-60,-50] #[-70,-60] #[-60,-40]
lon_range=[0,360] # [0,60]

ithr=0
for thresh in thresholds:
    second_thresh=second_thresholds[ithr]
    ds =euti.make_ds(fld=A.rho_epwp[:,:,:,:], lon=lon, lat=lat, zlev=zlev, time=time, 
                     thresh=thresh,second_thresh=second_thresh,zlev_event=zlev_event, 
                     lat_range=lat_range, lon_range=lon_range)
    ithr=ithr+1
    

    # get shape of varaiables
    nt,nz,ny,nx = np.shape( A.u )
    
    htopo_t = np.tile(A.htopo[None, :, :], ( nt, 1, 1))
    htopo_4D_x , time4D,lat4D,lon4D  = auti.cube4D_ds( event_ds=ds, aa=htopo_t , lon=lon, lat=lat, window=[0,5,5] , TZHkey='tyx', lat_range=[-999,999], lon_range=[-999,999] )
    
    htopo_MMM = auti.collapseSpace( htopo_4D_x , TZHkey='etyx')
    htopo_super_max = htopo_MMM[2].max( axis=1 )
    
    
    ########################################
    # Exclude events with topography nearby
    ########################################
    
    flat=np.where(htopo_super_max<0.0001)
    flat[0].shape
    
    ds_flat=ds.isel( index=flat[0] )    
    ds=ds_flat
    
    E_ = {'ds':ds }

    window=[3,2,2]
    #window=[6,2,2] # MPAS results are 3-hourly ...
    if ds.sizes['index'] < 75_000:
        window = [3,5,5] #[6,5,5]
        print( f"Big window ")
    #window=[6,5,5] # MPAS results are 3-hourly ...
    
    
    
    precl_4D , time4D,lat4D,lon4D  = auti.cube4D_ds( event_ds=ds, aa=A.precl , lon=lon, lat=lat, window=window , TZHkey='tzyx', lat_range=lat_range, lon_range=lon_range )
    htopo_4D , time4D,lat4D,lon4D  = auti.cube4D_ds( event_ds=ds, aa=htopo_t , lon=lon, lat=lat, window=window , TZHkey='tyx', lat_range=lat_range, lon_range=lon_range )
    zeta_4D, time4D,lat4D,lon4D = auti.cube4D_ds( event_ds=ds, aa=A.zeta , lon=lon, lat=lat, window=window , TZHkey='tzyx', lat_range=lat_range, lon_range=lon_range )
    tilt_4D, time4D,lat4D,lon4D = auti.cube4D_ds( event_ds=ds, aa=A.tilt , lon=lon, lat=lat, window=window , TZHkey='tzyx', lat_range=lat_range, lon_range=lon_range )
    fgf_4D, time4D,lat4D,lon4D  = auti.cube4D_ds( event_ds=ds, aa=A.fgf , lon=lon, lat=lat, window=window , TZHkey='tzyx', lat_range=lat_range, lon_range=lon_range )
    epwp_4D, time4D,lat4D,lon4D = auti.cube4D_ds( event_ds=ds, aa=A.rho_epwp , lon=lon, lat=lat, window=window , TZHkey='tzyx', lat_range=lat_range, lon_range=lon_range )
    upwp_4D, time4D,lat4D,lon4D = auti.cube4D_ds( event_ds=ds, aa=A.rho_upwp , lon=lon, lat=lat, window=window , TZHkey='tzyx', lat_range=lat_range, lon_range=lon_range )
    vpwp_4D, time4D,lat4D,lon4D = auti.cube4D_ds( event_ds=ds, aa=A.rho_vpwp , lon=lon, lat=lat, window=window , TZHkey='tzyx', lat_range=lat_range, lon_range=lon_range )
    u_4D, time4D,lat4D,lon4D    = auti.cube4D_ds( event_ds=ds, aa=A.u , lon=lon, lat=lat, window=window , TZHkey='tzyx', lat_range=lat_range, lon_range=lon_range )
    v_4D, time4D,lat4D,lon4D    = auti.cube4D_ds( event_ds=ds, aa=A.v , lon=lon, lat=lat, window=window , TZHkey='tzyx', lat_range=lat_range, lon_range=lon_range )
    th_4D, time4D,lat4D,lon4D   = auti.cube4D_ds( event_ds=ds, aa=A.th , lon=lon, lat=lat, window=window , TZHkey='tzyx', lat_range=lat_range, lon_range=lon_range )

    E_['time4D'], E_['lat4D'], E_['lon4D'] = time4D,lat4D,lon4D
    E_['u_4D'], E_['v_4D'], E_['htopo_4D'] = u_4D,v_4D,htopo_4D
    E_['zeta_4D'], E_['tilt_4D'] , E_['fgf_4D']  = zeta_4D,tilt_4D,fgf_4D
    E_['upwp_4D'], E_['vpwp_4D'] , E_['epwp_4D']  = upwp_4D,vpwp_4D,epwp_4D
    E_['precl_4D'], E_['th_4D'] = precl_4D,th_4D

    E = AttrDict( E_ )
    El.append(E)
    print( f" fininshed threshold = {thresh}. N events: Global= {E.ds.sizes['index']}, Region= {len(time4D)} " )

In [ ]:
zetalv=1.5e-5*np.linspace(-6,6,num=13)
ulv=np.linspace(-60,60,num=27)
thlv=np.concatenate( (270.+np.arange(11)*10 , 380.+ np.arange(11)*20) )   #np.linspace(270,600,num=27)
mflv=[0.001,0.002,.005, .01, .02]
Epls=[El[0], El[1], El[2], El[3] ] #, Eco_0 ]
print(thlv)
nxplo,nyplo=len( Epls ),1
fig,axs=plt.subplots( nyplo,nxplo , figsize=(nxplo*7+1,nyplo*8) )
axs=axs.flatten()
p=0

for Epl in Epls:
    nv,nt_v,nz_v,ny_v,nx_v = np.shape( Epl.zeta_4D )
    delta_time=3

    ax=axs[p]
    Epl_vv=euti.avg_over_v(Epl)
    zeta_poo=Epl_vv.zeta_4D.mean(axis=3)
    u_poo=Epl_vv.u_4D.mean(axis=3)
    th_poo=Epl_vv.th_4D.mean(axis=3)
    epwp_poo=Epl_vv.epwp_4D.mean(axis=3)
    colo = ax.contourf( np.arange(ny_v), zlev, zeta_poo[3,:,:], cmap='bwr' , levels=zetalv)
    lin1 = ax.contour( np.arange(ny_v), zlev, u_poo[3,:,:] , levels=ulv)
    ax.clabel(lin1, inline=True, fontsize=8, fmt='%1.0f')
    lin2 = ax.contour( np.arange(ny_v), zlev, th_poo[3,:,:] , levels=thlv, colors='red')
    ax.clabel(lin2, inline=True, fontsize=8, fmt='%1.0f')
    lin3 = ax.contour( np.arange(ny_v), zlev, epwp_poo[3,:,:] , levels=mflv, colors='black')
    ax.clabel(lin3, inline=True, fontsize=8, fmt='%1.3f')
    ax.set_ylim(0,20_000)
    p=p+1

cax = fig.add_axes([0.15, 0.02, 0.70, 0.03])
cbar = fig.colorbar(colo, cax=cax, orientation='horizontal')
cbar.set_label(f"vorticity  s{r'$^{-1}$' }")
plt.suptitle(  " Super Range on Latitude ",fontsize=36 )
"""
cax = fig.add_axes([0.15, -0.1, 0.70, 0.03])
cbar = fig.colorbar(linc, cax=cax, orientation='horizontal')
cbar.set_label('Some quantity')
"""

In [ ]:
El_0[3].keys()

In [ ]:

plt.plot(El_0[3].lat4D)
plt.plot(El[3].lat4D)


In [ ]:

print(El_0[3].zeta_4D.shape )
print(El[3].zeta_4D.shape )


In [ ]:

e=300
plt.plot( El_0[3].u_4D[e,3,:,5,5] )
plt.plot( El[3].u_4D[e,3,:,5,5] )


In [ ]:

scripfile='/glade/campaign/collections/gdex/data/d651077/cesmdata/inputdata/share/scripgrids/fv0.9x1.25_141008.nc'
Zog=xr.open_dataset( scripfile )

lat_ofcl,lon_ofcl= GrU.latlon(grid=None, scrip=scripfile ,Hkey='yx' ,get_area=False)

In [ ]:
print(lat_ofcl[0:8]-lat_ofcl[1:9] )
print(lon_ofcl[0:8]-lon_ofcl[1:9] )

print(0.5*(lat_ofcl[2]-lat_ofcl[0] ))
print(0.5*(lat_ofcl[3]-lat_ofcl[1] ))

dlon=lon_ofcl[1:-1]-lon_ofcl[0:-2]
dlat=lat_ofcl[1:-1]-lat_ofcl[0:-2]


In [ ]:
plt.plot( dlat , 'o')

In [ ]:
grid_lat=Zog.grid_center_lat.values

In [ ]:
print( grid_lat.min() , lat_ofcl[0] )

In [ ]:
plt.plot( grid_lat, '.' )
plt.ylim( -90,-60 )
plt.xlim(0,10_000)